# 04 — Error Analysis

This notebook investigates **where and why** each model fails — not just what the overall accuracy is.

Key questions:
1. Which sentiment class is hardest to predict?
2. What do high-confidence errors look like? (most dangerous failures)
3. Are there systematic linguistic patterns in errors (negation, sarcasm, short text)?
4. Where do both models agree and both fail?
5. What concrete improvements would address the most common failure modes?

> **Run `scripts/train_full.py` first** to generate models and processed data.

In [ ]:
import sys
from pathlib import Path

# Make project root importable
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from config import TWEET_EVAL_PATH, FIGURES_DIR, set_seed
set_seed()

LABEL_NAMES = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
COLORS = {0: '#e74c3c', 1: '#2ecc71', 2: '#3498db'}

print('Setup complete.')

## 1. Load Test Data

In [ ]:
from src.data_loader import preprocess_dataframe

df_raw = pd.read_csv(TWEET_EVAL_PATH)
df_raw = df_raw.dropna(subset=['label', 'text'])
df_raw['label'] = df_raw['label'].astype(int)

test_df = df_raw[df_raw['split'] == 'test'].copy().reset_index(drop=True)
test_df = preprocess_dataframe(test_df)

print(f'Test samples: {len(test_df)}')
print('Class distribution:')
print(test_df['label'].map(LABEL_NAMES).value_counts())

## 2. Baseline Model — Predictions on Test Set

In [ ]:
from src.baseline_model import load_baseline_model, predict as predict_baseline

baseline = load_baseline_model()
bl_labels, bl_probs = predict_baseline(baseline, test_df['clean_text'].tolist())
bl_conf = bl_probs.max(axis=1)

test_df['bl_pred']  = bl_labels
test_df['bl_conf']  = bl_conf
test_df['bl_correct'] = (bl_labels == test_df['label'].values)

bl_acc = test_df['bl_correct'].mean()
print(f'Baseline accuracy on test set: {bl_acc:.4f}')
print()
print(classification_report(test_df['label'], bl_labels, target_names=['Negative','Positive','Neutral']))

## 3. BERT Model — Predictions on Test Set

> Skip gracefully if BERT model is not yet trained.

In [ ]:
BERT_AVAILABLE = False
try:
    from src.bert_model import load_bert_model, predict_bert
    bert_model, bert_tokenizer = load_bert_model()
    bert_labels, bert_probs = predict_bert(bert_model, bert_tokenizer, test_df['clean_text'].tolist())
    bert_conf = bert_probs.max(axis=1)

    test_df['bert_pred']    = bert_labels
    test_df['bert_conf']    = bert_conf
    test_df['bert_correct'] = (bert_labels == test_df['label'].values)
    BERT_AVAILABLE = True

    bert_acc = test_df['bert_correct'].mean()
    print(f'BERT accuracy on test set: {bert_acc:.4f}')
    print()
    print(classification_report(test_df['label'], bert_labels, target_names=['Negative','Positive','Neutral']))
except Exception as e:
    print(f'BERT not available: {e}')
    print('Skipping BERT error analysis — run train_full.py --model bert first.')

## 4. Accuracy by Sentiment Class

In [ ]:
fig, axes = plt.subplots(1, 2 if BERT_AVAILABLE else 1, figsize=(12 if BERT_AVAILABLE else 6, 5))
if not BERT_AVAILABLE:
    axes = [axes]

for ax, (col, title) in zip(axes, [('bl_correct', 'Baseline (TF-IDF + LR)'),
                                    ('bert_correct', 'BERT Fine-tuned')][:2 if BERT_AVAILABLE else 1]):
    acc_by_class = test_df.groupby('label')[col].mean().reset_index()
    acc_by_class['class_name'] = acc_by_class['label'].map(LABEL_NAMES)
    bars = ax.bar(acc_by_class['class_name'], acc_by_class[col],
                  color=[COLORS[l] for l in acc_by_class['label']], edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, acc_by_class[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Accuracy')
    ax.set_xlabel('True Sentiment')
    ax.axhline(test_df[col].mean(), color='black', linestyle='--', alpha=0.5, label=f'Overall: {test_df[col].mean():.3f}')
    ax.legend()

fig.suptitle('Per-Class Accuracy on Test Set', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'error_accuracy_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Which class has the lowest accuracy?')
worst_class = test_df.groupby('label')['bl_correct'].mean().idxmin()
print(f'  Baseline worst class: {LABEL_NAMES[worst_class]} ({test_df.groupby("label")["bl_correct"].mean()[worst_class]:.3f})')

## 5. Confidence Distribution: Correct vs Incorrect Predictions

In [ ]:
n_plots = 2 if BERT_AVAILABLE else 1
fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
if not BERT_AVAILABLE:
    axes = [axes]

for ax, (conf_col, correct_col, title) in zip(
    axes,
    [('bl_conf', 'bl_correct', 'Baseline'), ('bert_conf', 'bert_correct', 'BERT')][:n_plots]
):
    correct_mask = test_df[correct_col]
    ax.hist(test_df.loc[correct_mask, conf_col], bins=30, alpha=0.6,
            color='#2ecc71', label='Correct', density=True)
    ax.hist(test_df.loc[~correct_mask, conf_col], bins=30, alpha=0.6,
            color='#e74c3c', label='Incorrect', density=True)
    ax.set_xlabel('Confidence (max softmax prob)')
    ax.set_ylabel('Density')
    ax.set_title(f'{title}: Confidence Distribution', fontsize=12)
    ax.legend()

    # Calibration note
    high_conf_wrong = ((test_df[conf_col] > 0.8) & ~test_df[correct_col]).sum()
    high_conf_total = (test_df[conf_col] > 0.8).sum()
    print(f'{title}: {high_conf_wrong}/{high_conf_total} high-confidence (>0.8) errors '
          f'= {high_conf_wrong/max(high_conf_total,1)*100:.1f}% of high-conf predictions wrong')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'error_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. High-Confidence Errors (Most Dangerous Failures)

These are the predictions where the model was **very confident but wrong** — the most concerning failures in any ML system.

In [ ]:
HIGH_CONF_THRESHOLD = 0.8

bl_high_conf_errors = test_df[
    (test_df['bl_conf'] > HIGH_CONF_THRESHOLD) & ~test_df['bl_correct']
].copy()
bl_high_conf_errors['true_label'] = bl_high_conf_errors['label'].map(LABEL_NAMES)
bl_high_conf_errors['pred_label'] = bl_high_conf_errors['bl_pred'].map(LABEL_NAMES)

print(f'Baseline: {len(bl_high_conf_errors)} high-confidence errors (confidence > {HIGH_CONF_THRESHOLD})')
print(f'Breakdown by true → predicted:')
print(bl_high_conf_errors.groupby(['true_label', 'pred_label']).size().reset_index(name='count').to_string(index=False))
print()
print('Sample high-confidence errors:')
display_cols = ['text', 'clean_text', 'true_label', 'pred_label', 'bl_conf']
print(bl_high_conf_errors[display_cols].head(10).to_string(index=False, max_colwidth=80))

In [ ]:
# Error pattern: true → predicted heatmap
bl_errors = test_df[~test_df['bl_correct']]
bl_cm = confusion_matrix(bl_errors['label'], bl_errors['bl_pred'], labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(7, 5))
labels_list = ['Negative', 'Positive', 'Neutral']
sns.heatmap(bl_cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=labels_list, yticklabels=labels_list, ax=ax)
ax.set_title('Baseline — Error Confusion Matrix\n(only incorrect predictions)', fontsize=12)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'error_confusion_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nMost common confusion: which true class gets predicted as which?')
for true_l in [0, 1, 2]:
    row = bl_cm[true_l]
    top_wrong = np.argsort(row)[::-1]
    top_wrong = [t for t in top_wrong if t != true_l and row[t] > 0]
    if top_wrong:
        print(f'  True {labels_list[true_l]} → most often predicted as {labels_list[top_wrong[0]]} ({row[top_wrong[0]]} times)')

## 7. Text Length vs Error Rate

In [ ]:
test_df['word_count'] = test_df['clean_text'].str.split().str.len()
test_df['length_bucket'] = pd.cut(test_df['word_count'],
                                   bins=[0, 5, 10, 20, 50, 200],
                                   labels=['1-5', '6-10', '11-20', '21-50', '50+'])

agg = test_df.groupby('length_bucket', observed=True).agg(
    count=('bl_correct', 'count'),
    bl_error_rate=('bl_correct', lambda x: 1 - x.mean()),
).reset_index()

if BERT_AVAILABLE:
    bert_agg = test_df.groupby('length_bucket', observed=True)['bert_correct'].apply(lambda x: 1 - x.mean()).reset_index()
    bert_agg.columns = ['length_bucket', 'bert_error_rate']
    agg = agg.merge(bert_agg, on='length_bucket')

print('Error rate by text length:')
print(agg.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(agg))
width = 0.35
bars1 = ax.bar(x - width/2 if BERT_AVAILABLE else x, agg['bl_error_rate'],
               width, label='Baseline', color='#e67e22', alpha=0.85)
if BERT_AVAILABLE:
    bars2 = ax.bar(x + width/2, agg['bert_error_rate'],
                   width, label='BERT', color='#8e44ad', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'{b}\n(n={c})' for b, c in zip(agg['length_bucket'], agg['count'])])
ax.set_xlabel('Word Count Bucket')
ax.set_ylabel('Error Rate')
ax.set_title('Error Rate by Text Length', fontsize=12)
ax.legend()
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'error_by_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Negation Patterns in Errors

A key failure mode for bag-of-words models: **negation** ("not good" classified as positive because 'good' has high weight).

In [ ]:
NEGATION_WORDS = ['not', 'no', 'never', "n't", 'neither', 'nor', 'cannot', "can't",
                  "won't", "doesn't", "didn't", "isn't", "wasn't", "aren't"]

def has_negation(text: str) -> bool:
    tokens = str(text).lower().split()
    return any(w in tokens for w in NEGATION_WORDS)

test_df['has_negation'] = test_df['text'].apply(has_negation)

neg_stats = test_df.groupby('has_negation').agg(
    count=('bl_correct', 'count'),
    bl_acc=('bl_correct', 'mean'),
).reset_index()
neg_stats.index = neg_stats['has_negation'].map({True: 'With negation', False: 'No negation'})

print('Baseline accuracy — negation vs no negation:')
print(neg_stats[['count', 'bl_acc']].rename(columns={'bl_acc': 'accuracy'}).to_string())

if BERT_AVAILABLE:
    neg_bert = test_df.groupby('has_negation')['bert_correct'].mean()
    print('\nBERT accuracy — negation vs no negation:')
    print(neg_bert.rename({True: 'With negation', False: 'No negation'}))

print('\nSample negation errors (true negative/positive, predicted opposite):')
neg_errors = test_df[
    test_df['has_negation'] &
    ~test_df['bl_correct'] &
    test_df['label'].isin([0, 1])  # binary sentiment, not neutral
][['text', 'label', 'bl_pred', 'bl_conf']].head(8)
neg_errors['true'] = neg_errors['label'].map(LABEL_NAMES)
neg_errors['pred'] = neg_errors['bl_pred'].map(LABEL_NAMES)
print(neg_errors[['text', 'true', 'pred', 'bl_conf']].to_string(index=False, max_colwidth=90))

## 9. SHAP — Most Misleading Tokens in Errors

Which tokens are causing the model to go wrong? We use SHAP to find the top-weighted tokens in high-confidence wrong predictions.

In [ ]:
try:
    from src.explain import explain_baseline_prediction

    print('Analysing SHAP for top high-confidence errors...')
    sample_errors = bl_high_conf_errors.head(5)

    for _, row in sample_errors.iterrows():
        contribs, pred_cls, _ = explain_baseline_prediction(baseline, row['clean_text'], n_top=5)
        true_lbl = LABEL_NAMES[row['label']]
        pred_lbl = LABEL_NAMES[row['bl_pred']]
        print(f'\nText: "{row["text"][:80]}"')
        print(f'True: {true_lbl}  |  Predicted: {pred_lbl}  |  Confidence: {row["bl_conf"]:.3f}')
        print('Top SHAP contributors:')
        for token, shap_val in contribs[:5]:
            direction = '→ positive' if shap_val > 0 else '→ negative'
            print(f'  "{token}": {shap_val:+.4f}  ({direction} push for predicted class)')
except ImportError:
    print('Install shap to enable this analysis: pip install shap')
except Exception as e:
    print(f'SHAP analysis failed: {e}')

## 10. Model Agreement Analysis

When **both** models agree and are wrong, those examples are the hardest — neither model can handle them.

In [ ]:
if BERT_AVAILABLE:
    test_df['both_agree'] = test_df['bl_pred'] == test_df['bert_pred']
    test_df['both_correct'] = test_df['bl_correct'] & test_df['bert_correct']
    test_df['both_wrong'] = ~test_df['bl_correct'] & ~test_df['bert_correct']
    test_df['baseline_only_correct'] = test_df['bl_correct'] & ~test_df['bert_correct']
    test_df['bert_only_correct'] = ~test_df['bl_correct'] & test_df['bert_correct']

    n = len(test_df)
    print('Agreement breakdown:')
    print(f'  Both correct:           {test_df["both_correct"].sum():5d} ({test_df["both_correct"].mean()*100:.1f}%)')
    print(f'  Both wrong:             {test_df["both_wrong"].sum():5d} ({test_df["both_wrong"].mean()*100:.1f}%)')
    print(f'  Baseline only correct:  {test_df["baseline_only_correct"].sum():5d} ({test_df["baseline_only_correct"].mean()*100:.1f}%)')
    print(f'  BERT only correct:      {test_df["bert_only_correct"].sum():5d} ({test_df["bert_only_correct"].mean()*100:.1f}%)')

    print('\nExamples where BOTH models fail:')
    hard_examples = test_df[test_df['both_wrong']].sample(min(5, test_df['both_wrong'].sum()), random_state=42)
    hard_examples['true'] = hard_examples['label'].map(LABEL_NAMES)
    hard_examples['bl_p'] = hard_examples['bl_pred'].map(LABEL_NAMES)
    hard_examples['bert_p'] = hard_examples['bert_pred'].map(LABEL_NAMES)
    print(hard_examples[['text', 'true', 'bl_p', 'bert_p']].to_string(index=False, max_colwidth=80))
else:
    print('BERT not available — skipping agreement analysis.')

## 11. Key Insights Summary

In [ ]:
print('=' * 65)
print('ERROR ANALYSIS — KEY INSIGHTS')
print('=' * 65)

# Class difficulty
bl_class_acc = test_df.groupby('label')['bl_correct'].mean()
worst_bl = bl_class_acc.idxmin()
print(f'\n1. HARDEST CLASS (Baseline):')
print(f'   {LABEL_NAMES[worst_bl]} — accuracy {bl_class_acc[worst_bl]:.3f}')
print(f'   This class is most often confused with: '
      + LABEL_NAMES[bl_high_conf_errors[bl_high_conf_errors["label"]==worst_bl]["bl_pred"].mode().iloc[0]
                    if len(bl_high_conf_errors[bl_high_conf_errors["label"]==worst_bl]) > 0
                    else list(LABEL_NAMES.keys())[0]])

# Negation
neg_acc = test_df[test_df['has_negation']]['bl_correct'].mean()
no_neg_acc = test_df[~test_df['has_negation']]['bl_correct'].mean()
print(f'\n2. NEGATION IMPACT:')
print(f'   Texts with negation:    accuracy = {neg_acc:.3f}')
print(f'   Texts without negation: accuracy = {no_neg_acc:.3f}')
print(f'   Drop = {(no_neg_acc - neg_acc):.3f} — bag-of-words misses negation context')

# High-confidence errors
print(f'\n3. HIGH-CONFIDENCE ERRORS (conf > {HIGH_CONF_THRESHOLD}):')
print(f'   Baseline: {len(bl_high_conf_errors)} errors with high confidence')
print(f'   These are the most dangerous — model is wrong and certain about it')

# Recommendations
print('\n4. RECOMMENDATIONS:')
print('   a) Negation handling: add bigrams ("not good") — already in config (1,2)')
print('      → Consider: character n-grams or negation-aware features')
print('   b) Neutral class: hardest for both models — boundary with pos/neg is fuzzy')
print('      → Consider: aspect-based SA to detect mixed sentiment')
print('   c) Short texts (<5 words): high error rate — insufficient signal')
print('      → Consider: minimum length filtering or separate short-text model')
if BERT_AVAILABLE:
    print('   d) Both-model failures: sarcasm and irony require world knowledge')
    print('      → Consider: larger LLM (e.g. RoBERTa, DeBERTa) or few-shot prompting')

print('=' * 65)

In [ ]:
# Save the enriched test DataFrame for reference
out_path = ROOT / 'reports' / 'error_analysis_test.csv'
test_df.to_csv(out_path, index=False)
print(f'Enriched test set saved → {out_path}')
print('(Not committed to git — gitignored with reports/)')